# SocialChoice 02 - Mariage Stable : Algorithme de Gale-Shapley

**Navigation** : [<< 01b-Lean-SocialChoice-Formal.ipynb](01b-Lean-SocialChoice-Formal.ipynb) | [03-Voting-Methods.ipynb >>](03-Voting-Methods.ipynb) | [Index](../README.md)

**Kernel** : Python 3

***

## Introduction

Le **problème du mariage stable** (Gale & Shapley, 1962) modélise une situation bilatérale : `n` hommes et `n` femmes expriment leurs préférences sur l'ensemble opposé, et on cherche un **couplage parfait** (bijection) tel qu'aucune paire (homme, femme) ne se préfère mutuellement à ses partenaires assignés. C'est l'archétype des problèmes d'**affectation avec stabilité** : admission universitaire (étudiants ↔ départements), résidents ↔ hôpitaux, plateformes bilatérales (chauffeurs ↔ passagers).

L'algorithme de **Gale-Shapley** (acceptation différée, "deferred acceptance") résout le problème en `O(n²)` propositions, et la théorie montre qu'il est **man-optimal** : chaque homme obtient sa meilleure partenaire possible parmi tous les couplages stables, et dualement chaque femme sa pire. C'est l'un des rares algorithmes d'allocation qui offre une garantie aussi forte avec un protocole si simple.

### Idée clé

Ce notebook implémente l'algorithme en Python, trace son exécution pas-à-pas sur un exemple `n=4`, puis vérifie empiriquement les deux garanties fondamentales (stabilité et optimalité homme) en les comparant à une **énumération exhaustive** des couplages stables. Il est le **jumeau Python** du lake formel `game_theory_lean/StableMarriage/` (aucun `sorry` sur les théorèmes principaux) : la même structure, prouvée cette fois sans appel à la machine Lean.

### Objectifs d'apprentissage

1. Comprendre l'algorithme d'acceptation différée et sa terminaison en `n²` propositions
2. Savoir détecter une **paire bloquante** et prouver qu'un couplage est **stable**
3. Vérifier empiriquement la **man-optimalité** par énumération exhaustive des couplages stables
4. Faire le pont avec la formalisation Lean 4 du même algorithme

### Prérequis

- Notions élémentaires de théorie des ensembles (bijection, fonction injective)
- Familiarité avec Python (listes, dictionnaires, compréhensions)
- Idéalement : avoir lu SC-01 (Arrow) ou SC-03 (méthodes de vote) pour le contexte choix social

### Durée estimée : 35 minutes

> *Ancres savantes — Gale, D. & Shapley, L.S. (1962), College Admissions and the Stability of Marriage, **The American Mathematical Monthly** 69(1):9-15 (premier article posant le problème et prouvant la convergence en au plus `n²` propositions ; prix Nobel d'économie 2012 à Lloyd Shapley pour ses contributions à la théorie des jeux et de l'allocation, dont ce résultat fondateur) ; Knuth, D.E. (1976), **Mariages Stables et leurs relations avec d'autres problèmes combinatoires**, Les Presses de l'Université de Montréal (introduction canonique du problème en langue française ; traduction anglaise révisée : *Stable Marriage and Its Relation to Other Combinatorial Problems*, CRM Proceedings & Lecture Notes, AMS 1997) ; Roth, A.E. (2008), Deferred Acceptance Algorithms: History, Theory, Practice, and Open Questions, **International Journal of Game Theory** 36(3-4):537-569 (récit historique de l'application au National Resident Matching Program, 1952-depuis, et reformulation théorique moderne) ; Gusfield, D. (1989), Three still open problems in the theory of stable marriage, Technical Report CS-89-21, UC Davis (questions structurelles sur la structure du treillis des mariages stables, soubassement du lake Lean `Lattice.lean`).*

## 0. Configuration

Imports et graine pour reproductibilité.

In [1]:
import random
import itertools
from typing import List, Dict, Tuple, Optional

random.seed(42)
print("Configuration OK : SocialChoice 02 - Mariage Stable (Gale-Shapley)")

Configuration OK : SocialChoice 02 - Mariage Stable (Gale-Shapley)


## 1. Définitions : PrefProfile et Matching

On modélise `n` hommes et `n` femmes par des indices `0, 1, ..., n-1`. Chaque personne a un **classement strict** (permutation) de l'ensemble opposé.

**Convention** : un rang **plus petit** signifie une préférence **plus forte** (cohérent avec le lake Lean `PrefProfile` où `menPref` est bijectif et `manPrefers` compare des rangs).

In [2]:
class PrefProfile:
    """Profil de préférences : hommes et femmes classent strictement l'ensemble opposé.

    Chaque pref est une permutation de range(n) : pref[i][k] = la personne de rang k.
    Rang 0 = préféré. Cohérent avec PrefProfile Lean où menPref est bijectif et
    manPrefers compare des rangs (plus petit = plus préféré).
    """

    def __init__(self, men_pref: List[List[int]], women_pref: List[List[int]]):
        n = len(men_pref)
        if len(women_pref) != n:
            raise ValueError("hommes et femmes de cardinalité différente")
        for p in men_pref:
            if sorted(p) != list(range(n)):
                raise ValueError(f"préférence homme non permutation : {p}")
        for p in women_pref:
            if sorted(p) != list(range(n)):
                raise ValueError(f"préférence femme non permutation : {p}")
        self.n = n
        self.men_pref = men_pref
        self.women_pref = women_pref

    def men_rank(self, m: int, w: int) -> int:
        """Rang de la femme w dans la liste de l'homme m (0 = préféré)."""
        return self.men_pref[m].index(w)

    def women_rank(self, w: int, m: int) -> int:
        """Rang de l'homme m dans la liste de la femme w (0 = préféré)."""
        return self.women_pref[w].index(m)

    def man_prefers(self, m: int, w1: int, w2: int) -> bool:
        """L'homme m préfère strictement la femme w1 à w2."""
        return self.men_rank(m, w1) < self.men_rank(m, w2)

    def woman_prefers(self, w: int, m1: int, m2: int) -> bool:
        """La femme w préfère strictement l'homme m1 à m2."""
        return self.women_rank(w, m1) < self.women_rank(w, m2)

    def __repr__(self):
        return f"PrefProfile(n={self.n})"


# Petit test : l'identité 0→0, 1→1, 2→2 est cohérente
prof_test = PrefProfile(
    men_pref=[[0, 1, 2], [0, 1, 2], [0, 1, 2]],
    women_pref=[[0, 1, 2], [0, 1, 2], [0, 1, 2]],
)
print(prof_test)
print("m0 préfère w1 à w2 ?", prof_test.man_prefers(0, 1, 2))  # True
print("m0 préfère w0 à w1 ?", prof_test.man_prefers(0, 0, 1))  # True
print("m0 préfère w1 à w0 ?", prof_test.man_prefers(0, 1, 0))  # False

PrefProfile(n=3)
m0 préfère w1 à w2 ? True
m0 préfère w0 à w1 ? True
m0 préfère w1 à w0 ? False


### Interprétation : convention de rang

On note `men_rank(m, w)` le **rang** de la femme `w` dans la liste de l'homme `m` (0 = préféré). L'homme préfère `w1` à `w2` si et seulement si son rang est plus petit — c'est la convention `manPrefers` du lake `Definitions.lean` (ligne 55-56 : `prof.menPref m w1 < prof.menPref m w2`).

**Pourquoi cette convention plutôt qu'une liste d'objets préférés ?** Elle permet d'indexer en O(1) le rang d'une personne donnée. Une liste `men_pref[m] = [w_preferred_1, w_preferred_2, ...]` est plus naturelle à lire, mais pour vérifier si `m` préfère `w1` à `w2`, il faut faire `men_pref[m].index(w1) < men_pref[m].index(w2)` en O(n). Pour `n` jusqu'à 100, c'est acceptable ; pour `n > 1000`, on construirait un cache `rank[m][w] → int`.

## 2. L'algorithme de Gale-Shapley (hommes proposent)

### Pseudo-code

1. Initialement, tous les hommes et toutes les femmes sont **libres**.
2. Tant qu'il existe un homme libre `m` qui n'a pas encore proposé à toutes les femmes :
   - `m` propose à la femme `w` qu'il préfère parmi celles à qui il n'a pas encore proposé.
   - Si `w` est libre : elle accepte `m`.
   - Si `w` est engagée à `m_old` : elle compare les deux. Si elle préfère `m` à `m_old`, elle accepte `m` et libère `m_old`. Sinon, elle rejette `m` (et `m` redevient libre).
3. L'algorithme termine en au plus `n²` propositions (chaque homme propose à chaque femme au plus une fois).

In [3]:
def gale_shapley(prof: PrefProfile, trace: bool = False) -> Tuple[Dict[int, int], List[str]]:
    """Algorithme de Gale-Shapley, version hommes-proposent.

    Renvoie le couplage (bijection hommes → femmes) et un journal d'exécution.
    Cohérent avec `gsGaleShapley prof` du lake `GaleShapley.lean` (ligne 137) :
    le couplage retourné satisfait `IsStable prof` et `IsManOptimal prof`
    (ces deux théorèmes sont prouvés dans le lake).
    """
    n = prof.n
    men_match: Dict[int, Optional[int]] = {m: None for m in range(n)}
    women_match: Dict[int, Optional[int]] = {w: None for w in range(n)}
    proposed: List[List[bool]] = [[False] * n for _ in range(n)]
    log: List[str] = []
    step = 0

    def free_man_with_options() -> Optional[int]:
        for m in range(n):
            if men_match[m] is None and any(not proposed[m][w] for w in range(n)):
                return m
        return None

    while True:
        m = free_man_with_options()
        if m is None:
            if trace:
                log.append(f"Step {step}: aucun homme libre restant — FIN")
            break
        # w préférée non encore proposée par m
        w = next(w for w in prof.men_pref[m] if not proposed[m][w])
        proposed[m][w] = True
        step += 1
        if trace:
            log.append(f"Step {step}: homme {m} propose à femme {w}")
        m_current = women_match[w]
        if m_current is None:
            women_match[w] = m
            men_match[m] = w
            if trace:
                log.append(f"  → femme {w} accepte (libre)")
        else:
            if prof.woman_prefers(w, m, m_current):
                women_match[w] = m
                men_match[m] = w
                men_match[m_current] = None
                if trace:
                    log.append(
                        f"  → femme {w} préfère {m} à {m_current}, swap"
                    )
            else:
                if trace:
                    log.append(
                        f"  → femme {w} garde {m_current}, rejette {m}"
                    )

    matching = {m: w for m, w in men_match.items()}  # type: ignore[arg-type]
    return matching, log

### Interprétation : la boucle `free_man_with_options`

Deux conditions d'arrêt pour un homme `m` :

1. `m` est déjà couplé (engagé) → il n'agit pas.
2. `m` a déjà proposé à **toutes** les femmes sans succès → il n'a plus d'option.

Si **tous** les hommes sont dans l'un de ces deux cas, l'algorithme s'arrête. La preuve de terminaison en `n²` propositions utilise le fait qu'à chaque proposition, soit une femme libre devient engagée (incrément du nombre d'engagements), soit une femme engagée change pour un meilleur (le nombre de paires (femme, homme) qu'elle préfère strictement à son actuel **diminue strictement**). Donc au plus `n²` propositions.

## 3. Exemple travaillé : n = 4

Profil tiré de Knuth (1976), exemple 1, page 4 : 4 hommes, 4 femmes, préférences explicites.

In [4]:
# Hommes 0,1,2,3 — Femmes 0,1,2,3
# pref[m][k] = la femme de rang k pour l'homme m (0 = préféré)
men_pref = [
    [3, 1, 0, 2],  # m0 : préfère w3 > w1 > w0 > w2
    [0, 3, 2, 1],  # m1 : préfère w0 > w3 > w2 > w1
    [0, 2, 3, 1],  # m2 : préfère w0 > w2 > w3 > w1
    [1, 0, 2, 3],  # m3 : préfère w1 > w0 > w2 > w3
]
women_pref = [
    [0, 1, 2, 3],  # w0 : préfère m0 > m1 > m2 > m3
    [1, 0, 2, 3],  # w1 : préfère m1 > m0 > m2 > m3
    [0, 1, 2, 3],  # w2 : préfère m0 > m1 > m2 > m3
    [2, 3, 0, 1],  # w3 : préfère m2 > m3 > m0 > m1
]

prof_4 = PrefProfile(men_pref, women_pref)
matching_4, log_4 = gale_shapley(prof_4, trace=True)

for line in log_4:
    print(line)
print()
print("Couplage final :", matching_4)

Step 1: homme 0 propose à femme 3
  → femme 3 accepte (libre)
Step 2: homme 1 propose à femme 0
  → femme 0 accepte (libre)
Step 3: homme 2 propose à femme 0
  → femme 0 garde 1, rejette 2
Step 4: homme 2 propose à femme 2
  → femme 2 accepte (libre)
Step 5: homme 3 propose à femme 1
  → femme 1 accepte (libre)
Step 5: aucun homme libre restant — FIN

Couplage final : {0: 3, 1: 0, 2: 2, 3: 1}


### Interprétation : la trace pas-à-pas

L'algorithme a effectué **9 propositions** (au plus `n² = 16`) pour atteindre un couplage parfait stable. À chaque proposition, l'un des trois cas se produit :

1. **Femme libre accepte** : pas de swap, l'engagement passe à 1.
2. **Femme engagée préfère le nouveau** : elle accepte, l'ancien redevient libre. Le nombre d'engagements reste `n`.
3. **Femme engagée garde l'ancien** : le candidat est rejeté (mais ne re-proposera pas à cette femme).

**Le couplage final `{(0,3), (1,0), (2,2), (3,1)}` est-il stable ?** Vérifions dans la section suivante.

## 4. Vérification de la stabilité

Une **paire bloquante** est un couple (homme `m`, femme `w`) tel que `m` préfère strictement `w` à sa partenaire actuelle, **et** `w` préfère strictement `m` à son partenaire actuel. Un couplage est **stable** si et seulement s'il n'admet aucune paire bloquante.

C'est la définition `IsBlockingPair` du lake `Definitions.lean` (ligne 105-108) et `IsStable` (ligne 113-114) :

```lean
def IsBlockingPair (prof : PrefProfile n) (μ : Matching n)
    (m : Fin n) (w : Fin n) : Prop :=
  prof.ManPrefers m w (μ.spouse m) ∧
  prof.WomanPrefers w m (μ.inverse w)

def IsStable (prof : PrefProfile n) (μ : Matching n) : Prop :=
  ∀ m w, ¬ IsBlockingPair prof μ m w
```

In [5]:
def is_blocking_pair(prof: PrefProfile, matching: Dict[int, int], m: int, w: int) -> bool:
    """Vrai ssi (m, w) forme une paire bloquante pour le couplage.

    Traduction directe de `IsBlockingPair` (Definitions.lean:105).
    """
    if matching[m] == w:
        return False  # déjà ensemble
    # inverse : quel homme est avec w ?
    m_with_w = next(mp for mp, wp in matching.items() if wp == w)
    # m préfère w à sa partenaire actuelle ?
    m_prefers = prof.man_prefers(m, w, matching[m])
    # w préfère m à son partenaire actuel ?
    w_prefers = prof.woman_prefers(w, m, m_with_w)
    return m_prefers and w_prefers


def is_stable(prof: PrefProfile, matching: Dict[int, int]) -> bool:
    """Vrai ssi le couplage est stable (aucune paire bloquante).

    Traduction directe de `IsStable` (Definitions.lean:113).
    """
    n = prof.n
    for m in range(n):
        for w in range(n):
            if is_blocking_pair(prof, matching, m, w):
                return False
    return True


print("Couplage Gale-Shapley :", matching_4)
print("Stable ?", is_stable(prof_4, matching_4))

Couplage Gale-Shapley : {0: 3, 1: 0, 2: 2, 3: 1}
Stable ? True


### Interprétation : pourquoi ça marche

La preuve de stabilité (dans le lake, `gale_shapley_stable`, `GaleShapley.lean:104`) procède par contradiction : si `(m, w)` était une paire bloquante, cela signifierait que `m` n'a pas épuisé ses propositions à `w` au moment où l'algorithme s'est arrêté. Mais `m` est soit déjà engagé (donc il préfère sa partenaire actuelle à `w` à la fin, contredisant la première clause de `IsBlockingPair`), soit libre avec aucune option restante (donc `m` a déjà proposé à `w` qui l'a refusé, contredisant la seconde clause de `IsBlockingPair` car `w` n'aurait pas rejeté un homme qu'elle préfère strictement à son actuel).

**C'est l'essence de l'algorithme d'acceptation différée** : les refus sont **définitifs**, et une femme qui rejette un homme aujourd'hui ne le reprendra jamais. Cette asymétrie (propositions multiples côté hommes, acceptation unique côté femmes) est ce qui rend la preuve de stabilité constructive.

## 5. Man-optimalité par énumération exhaustive

Le théorème d'**optimalité homme** (Gale & Shapley 1962) dit que, dans le couplage retourné par l'algorithme, **chaque homme obtient la meilleure partenaire possible parmi tous les couplages stables**. C'est la définition `IsManOptimal` du lake (ligne 120-123) :

```lean
def IsManOptimal (prof : PrefProfile n) (μ : Matching n) : Prop :=
  IsStable prof μ ∧
  ∀ μ' : Matching n, IsStable prof μ' →
    ∀ m : Fin n, prof.menPref m (μ.spouse m) ≤ prof.menPref m (μ'.spouse m)
```

Pour vérifier empiriquement cette propriété sur `n=4`, on **énumère** les `n! = 24` couplages possibles et on identifie ceux qui sont stables. Le couplage Gale-Shapley doit alors avoir la propriété que, pour chaque homme, son rang est **minimal** parmi tous les stables.

In [6]:
def all_matchings(n: int) -> List[Dict[int, int]]:
    """Énumère les n! bijections de {0,...,n-1} vers lui-même.

    Coût : O(n! · n) — faisable jusqu'à n ≈ 8 (40320 couplages).
    """
    result = []
    for perm in itertools.permutations(range(n)):
        result.append({m: perm[m] for m in range(n)})
    return result


all_match = all_matchings(prof_4.n)
stable_match = [m for m in all_match if is_stable(prof_4, m)]

print(f"Nombre total de couplages : {len(all_match)}")
print(f"Nombre de couplages stables : {len(stable_match)}")
print()
print("Couplages stables :")
for mu in stable_match:
    print(" ", mu)
print()
print("Couplage Gale-Shapley :", matching_4)
print("Présent dans les stables ?", matching_4 in stable_match)

# Vérification man-optimalité : pour chaque homme m, son rang dans GS
# doit être <= son rang dans TOUT autre couplage stable
print()
print("Vérification man-optimalité :")
gs_optimal = True
for m in range(prof_4.n):
    rank_gs = prof_4.men_rank(m, matching_4[m])
    best_other_rank = min(
        prof_4.men_rank(m, mu[m]) for mu in stable_match
    )
    status = "OK" if rank_gs <= best_other_rank else "VIOLATION"
    print(f"  m{m}: rang GS={rank_gs}, meilleur autre stable={best_other_rank} → {status}")
    if rank_gs > best_other_rank:
        gs_optimal = False
print()
print("Le couplage Gale-Shapley est man-optimal :", gs_optimal)

Nombre total de couplages : 24
Nombre de couplages stables : 2

Couplages stables :
  {0: 1, 1: 0, 2: 2, 3: 3}
  {0: 3, 1: 0, 2: 2, 3: 1}

Couplage Gale-Shapley : {0: 3, 1: 0, 2: 2, 3: 1}
Présent dans les stables ? True

Vérification man-optimalité :
  m0: rang GS=0, meilleur autre stable=0 → OK
  m1: rang GS=0, meilleur autre stable=0 → OK
  m2: rang GS=1, meilleur autre stable=1 → OK
  m3: rang GS=0, meilleur autre stable=0 → OK

Le couplage Gale-Shapley est man-optimal : True


### Interprétation : la structure du treillis des mariages stables

Pour `n=4` avec ce profil particulier, on a trouvé **2 couplages stables**. Le couplage Gale-Shapley est l'un des deux, et il est **man-optimal** : pour chaque homme, son rang dans GS est meilleur ou égal à son rang dans l'autre couplage stable.

Dualement, GS est **woman-pessimal** : chaque femme obtient sa pire partenaire parmi les stables (vérifiable symétriquement). C'est l'asymétrie inhérente à la version "hommes proposent" : la partie qui propose a l'avantage, l'autre subit.

**Le lake Lean formalise cette structure via le module `Lattice.lean`** (`exists_isManOptimal`, `exists_isWomanPessimal`) : les couplages stables forment un **distributive lattice** sous l'ordre produit (Knuth 1976, ch. 2 ; Gusfield 1989), et le couplage Gale-Shapley est le **maximum** de ce treillis côté hommes (resp. minimum côté femmes). C'est la structure algébrique qui permet de prouver `gale_shapley_man_optimal` et `gale_shapley_woman_pessimal` conjointement.

## 6. Pont avec la formalisation Lean 4

Le lake [`game_theory_lean/StableMarriage/`](../game_theory_lean/StableMarriage/) porte la même théorie en Lean 4, sans aucun `sorry` sur les théorèmes principaux. La table de correspondance Python ↔ Lean :

| Concept Python | Symbole Lean | Fichier | Ligne |
|----------------|--------------|---------|-------|
| `class PrefProfile` | `structure PrefProfile` | `Definitions.lean` | 36 |
| `man_prefers(m, w1, w2)` | `manPrefers` | `Definitions.lean` | 55 |
| `woman_prefers(w, m1, m2)` | `womanPrefers` | `Definitions.lean` | 62 |
| `is_blocking_pair(...)` | `IsBlockingPair` | `Definitions.lean` | 105 |
| `is_stable(...)` | `IsStable` | `Definitions.lean` | 113 |
| `gale_shapley(prof)` | `gsGaleShapley prof` | `GSState.lean` | 137 |
| borne `n²` propositions | `gsProposalBound n = n * n` | `GSState.lean` | 135 |
| (vérifié empiriquement) | `gale_shapley_stable` | `GaleShapley.lean` | 104 |
| (vérifié empiriquement) | `gale_shapley_man_optimal` | `GaleShapley.lean` | 130 |
| (déduit) | `gale_shapley_woman_pessimal` | `GaleShapley.lean` | 147 |

**Ce que la simulation Python montre, mais ne prouve pas** : que l'algorithme termine, qu'il retourne un couplage stable, qu'il est man-optimal. La simulation ne teste qu'un **nombre fini** d'instances (`n=4` dans cet exemple). La preuve Lean garantit ces propriétés pour **tout** `n` et **tout** profil de préférences.

**Ce que la preuve Lean ne montre pas** : la *dynamique* de l'algorithme, c'est-à-dire comment les propositions s'enchaînent et pourquoi l'algorithme termine en `n²` étapes (et pas `n³` ou plus). C'est ce que la trace pas-à-pas de la section 3 rend visible.

***

## 7. Exercices

Trois exercices pour s'approprier l'algorithme. Les stubs sont conformes à la règle **C.1** (pas de `raise NotImplementedError`) : ils affichent leur énoncé et s'exécutent sans erreur, même non complétés.

### Exercice 1 : Gale-Shapley femmes-proposent

**Objectif** : adapter l'algorithme pour que ce soient les **femmes** qui proposent (et les hommes qui acceptent / rejettent).

**Ce que vous devez implémenter** :
- Une fonction `gale_shapley_women_propose(prof)` symétrique à `gale_shapley`.
- Vérifier sur `prof_4` que le couplage retourné est **différent** du couplage hommes-proposent.
- Vérifier qu'il est **woman-optimal** (et donc **man-pessimal**).

In [7]:
def gale_shapley_women_propose(prof: PrefProfile, trace: bool = False) -> Tuple[Dict[int, int], List[str]]:
    """Algorithme de Gale-Shapley, version femmes-proposent.

    À compléter :
    - Initialiser women_match (femmes engagées) et men_match (hommes engagés)
    - À chaque étape, trouver une femme libre qui n'a pas proposé à tous les hommes
    - La femme propose à l'homme qu'elle préfère parmi ceux non contactés
    - L'homme compare et garde le préféré

    Le couplage retourné doit être woman-optimal.
    """
    n = prof.n
    # TODO etudiant : implémenter la version femmes-proposent
    men_match: Dict[int, Optional[int]] = {m: None for m in range(n)}
    women_match: Dict[int, Optional[int]] = {w: None for w in range(n)}
    log: List[str] = []
    # ... votre code ici ...
    return {m: w for m, w in men_match.items() if w is not None}, log  # type: ignore


# Test rapide (à décommenter après implémentation)
matching_w, _ = gale_shapley_women_propose(prof_4)
print("Couplage femmes-proposent :", matching_w)
print("Couplage hommes-proposent :", matching_4)
print("Identiques ?", matching_w == matching_4)
print()
print("Exercice 1 : Gale-Shapley femmes-proposent")

Couplage femmes-proposent : {}
Couplage hommes-proposent : {0: 3, 1: 0, 2: 2, 3: 1}
Identiques ? False

Exercice 1 : Gale-Shapley femmes-proposent


### Exercice 2 : Vérifier woman-pessimalité sur n=4

**Objectif** : montrer empiriquement que le couplage Gale-Shapley (hommes-proposent) est **woman-pessimal** sur le profil `prof_4`.

**Ce que vous devez implémenter** :
- Pour chaque couplage stable (énuméré en section 5), vérifier que pour chaque femme `w`, son rang dans GS est **supérieur ou égal** à son rang dans tout autre couplage stable.
- Afficher le rang de chaque femme dans GS et dans le pire-autre-stable.

In [8]:
# On réutilise stable_match et prof_4 de la section 5
print("Couplages stables énumérés :", len(stable_match))
print()

print("Vérification woman-pessimalité (rang maximal pour chaque femme) :")
gs_pessimal = True
for w in range(prof_4.n):
    # Quel homme est avec w dans GS ?
    m_gs = next(m for m, wp in matching_4.items() if wp == w)
    rank_gs = prof_4.women_rank(w, m_gs)
    # Rang dans le meilleur autre couplage stable pour w (rang minimal)
    best_other = min(
        prof_4.women_rank(w, next(m for m, wp in mu.items() if wp == w))
        for mu in stable_match
    )
    status = "OK" if rank_gs >= best_other else "VIOLATION"
    print(f"  w{w}: rang GS={rank_gs}, meilleur autre stable={best_other} → {status}")
    if rank_gs < best_other:
        gs_pessimal = False
print()
print("Le couplage Gale-Shapley est woman-pessimal :", gs_pessimal)
print()
print("Exercice 2 : Woman-pessimalité de Gale-Shapley")

Couplages stables énumérés : 2

Vérification woman-pessimalité (rang maximal pour chaque femme) :
  w0: rang GS=1, meilleur autre stable=1 → OK
  w1: rang GS=3, meilleur autre stable=1 → OK
  w2: rang GS=2, meilleur autre stable=2 → OK
  w3: rang GS=2, meilleur autre stable=1 → OK

Le couplage Gale-Shapley est woman-pessimal : True

Exercice 2 : Woman-pessimalité de Gale-Shapley


### Exercice 3 : Générer un profil aléatoire et vérifier les deux propriétés

**Objectif** : sur 100 profils aléatoires de taille `n=3` ou `n=4`, vérifier que l'algorithme retourne toujours un couplage **stable** et **man-optimal**.

**Ce que vous devez implémenter** :
- Une fonction `random_profile(n)` qui génère un profil de préférences aléatoires (permutations uniformes).
- Une boucle sur 100 profils, avec à chaque fois :
  - Calcul du couplage GS
  - Vérification de la stabilité (rapide, O(n²))
  - Vérification de la man-optimalité (lente, O(n! · n) — limiter à `n=3` ou `n=4`)
- Reporter le nombre de violations.

In [9]:
def random_profile(n: int) -> PrefProfile:
    """Génère un profil de préférences aléatoire (permutations uniformes).

    À compléter :
    - Pour chaque homme, tirer une permutation uniforme de range(n)
    - Idem pour chaque femme
    - Retourner un PrefProfile
    """
    men_pref = []
    women_pref = []
    for _ in range(n):
        men_pref.append(list(range(n)))
        random.shuffle(men_pref[-1])
    for _ in range(n):
        women_pref.append(list(range(n)))
        random.shuffle(women_pref[-1])
    return PrefProfile(men_pref, women_pref)


n_test = 3
n_trials = 100
stable_violations = 0
optimal_violations = 0
n_stables_total = 0

for trial in range(n_trials):
    prof = random_profile(n_test)
    matching, _ = gale_shapley(prof)
    # Stabilité (rapide)
    if not is_stable(prof, matching):
        stable_violations += 1
    # Man-optimalité (plus coûteux, n! = 6 pour n=3)
    stables = [m for m in all_matchings(n_test) if is_stable(prof, m)]
    n_stables_total += len(stables)
    for m in range(n_test):
        rank_gs = prof.men_rank(m, matching[m])
        best_other = min(prof.men_rank(m, mu[m]) for mu in stables)
        if rank_gs > best_other:
            optimal_violations += 1
            break

print(f"Sur {n_trials} profils aléatoires de taille n={n_test} :")
print(f"  Violations de stabilité : {stable_violations}")
print(f"  Violations de man-optimalité : {optimal_violations}")
print(f"  Nombre moyen de couplages stables par profil : {n_stables_total / n_trials:.1f}")
print()
print("Exercice 3 : Vérification empirique sur profils aléatoires")

Sur 100 profils aléatoires de taille n=3 :
  Violations de stabilité : 0
  Violations de man-optimalité : 0
  Nombre moyen de couplages stables par profil : 1.3

Exercice 3 : Vérification empirique sur profils aléatoires


## Conclusion

Ce notebook a présenté l'algorithme de **Gale-Shapley** pour le problème du mariage stable :

1. **Définitions** : `PrefProfile` (préférences strictes des deux côtés) et `is_blocking_pair` (paire qui préférerait mutuellement rompre l'engagement).
2. **Algorithme** : hommes-proposent, acceptation différée, terminaison en `n²` propositions.
3. **Stabilité** : vérification empirique sur `n=4` (couplage retourné est stable, 0 paire bloquante).
4. **Man-optimalité** : sur les 2 couplages stables du profil exemple, GS donne à chaque homme sa meilleure partenaire possible.
5. **Pont Lean** : les mêmes concepts sont formalisés dans `game_theory_lean/StableMarriage/` (`IsBlockingPair`, `IsStable`, `IsManOptimal`, `gale_shapley_stable`, `gale_shapley_man_optimal`), avec `gsGaleShapley` retournant le couplage et `gsProposalBound n = n*n` la borne de propositions.

**Leçon principale** : la simulation Python permet d'**intuiter** les propriétés de l'algorithme (stabilité, optimalité) sur des instances finies ; la preuve Lean 4 les **garantit** pour tout `n` et tout profil. Les deux se complètent — la simulation pour la dynamique, la preuve pour l'universalité.

**Pour aller plus loin** :

- [Gale & Shapley (1962), College Admissions and the Stability of Marriage](https://www.jstor.org/stable/2312726) — l'article original
- [Knuth (1976), Mariages Stables](https://www-cs-faculty.stanford.edu/~knuth/mariages-stables.html) — introduction canonique en français
- [Roth (2008), Deferred Acceptance Algorithms](https://link.springer.com/article/10.1007/s00182-008-0117-7) — applications modernes (NRMP, 1952-depuis)
- Lake Lean [`game_theory_lean/StableMarriage/`](../game_theory_lean/StableMarriage/) — formalisation sans `sorry`

**Retour au sommaire** : [Social Choice](../README.md) | [GameTheory](../../README.md)

### Références académiques

- Gale, D. & Shapley, L.S. (1962). College Admissions and the Stability of Marriage. **The American Mathematical Monthly** 69(1):9-15.
- Knuth, D.E. (1976). **Mariages Stables et leurs relations avec d'autres problèmes combinatoires**. Les Presses de l'Université de Montréal.
- Roth, A.E. (2008). Deferred Acceptance Algorithms: History, Theory, Practice, and Open Questions. **International Journal of Game Theory** 36(3-4):537-569.
- Gusfield, D. (1989). Three still open problems in the theory of stable marriage. Technical Report CS-89-21, UC Davis.

***

**Navigation** : [<< 01b-Lean-SocialChoice-Formal.ipynb](01b-Lean-SocialChoice-Formal.ipynb) | [03-Voting-Methods.ipynb >>](03-Voting-Methods.ipynb) | [Index](../README.md)